# Special Flood Inundation Mapping

This notebook demonstrates how to create special maps with a FLDPLN library. Those maps include:
* Flood depth map with a constant stage at all the flood source pixels (FSPs, i.e., stream pixels)
* 'MinDtf' map, the minimum stage (or depth to flood (DTF)) needed to inundate a floodplain pixel (FPP)
* 'NumOfFsps' map, the number of FSPs that may inundate a FPP
* 'Depression' map, the filled depression depth 

## Import Modules and Packages

In [1]:
import time

# import the mapping module
from fldpln.mapping import *

## Setup Tiled Library and Output Map Folders

Here we setup the folder under which tiled libraries (organized as sub-folders) are located. We also setup the output folder under which a map sub-folder and a 'scratch' sub-folder will be created. The map folder, which is specified later, contains all inundation depth maps for the libraries. The scratch folder stores temporary files during the mapping. We can create the special maps for a set of tiled libraries and we assume that those libraries are located under the same folder. It is such designed as **libraries may overlap in space**, and tiles can be uniquely identified by the combination of library name and tile ID.

In [2]:
# folder that may have more than one tiled libraries
libFolder =  'E:/fldpln/workshop/devcon26/wildcat_10m_3dep' # wildcat

# Set mapping output folder
outputFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps'

## Setup Mapping Parameters

There are several parameters need to be set before the maps are generated. Those include:
* The tiled libraries to be mapped (see variable libs2Map)
* What special map to be created (i.e., specialDof)
* The name of the special map folder (i.e., outMapFolderName) which is under the output folder and stores all inundation depth maps as COG GeoTIFFs
* Whether to mosaic the tiles as single COG GeoTIFF file and whether to use a Dask local cluster for parallel mapping

In [3]:
# select the libraries under tiled library folder to generate maps
libs2Map = ['tiledlib']

# set the special map wanted
# 'MinDtf','NumOfFsps','Depression', or a real number (for example 20.5, in foot for KS libraries) representing constant DOF at all the FSPs
specialDof = 'MinDtf' # Wildcat 3DEP DEM has a vertical unit of meters

# set up output map-folder under the outputFolder
outMapFolderName = 'mindtf'

# Create folders for storing temp and output map files
outMapFolder,scratchFolder = CreateFolders(outputFolder,'scratch',outMapFolderName)

# whether mosaic tiles as a single COG
mosaicTiles = True #True #False

## Generate Special Map

In [4]:
# show mapping info
print(f'Tiled FLDPLN library folder: {libFolder}')
print(f'Map folder: {outMapFolder}')
# libs to map
print(f'Libraries to map: {libs2Map}')

# check running time
startTimeAllLibs = time.time()

# dict to store lib processing time
libTime={}

# map each library
for libName in libs2Map:
    # check running time
    startTime = time.time()
    
    print(f'Map {libName} ...')
    tileTifs = MapFloodDepthWithTiles(libFolder,libName,'snappy',outMapFolder,fspDof=specialDof,aoiExtent=None)
    print(f'Actual mapped tiles: {tileTifs}')

    # Mosaic all the tiles from a library into one tif file
    if mosaicTiles and not(tileTifs is None):
        print('Mosaic tile maps ...')
        mosaicTifName = libName+'_'+outMapFolderName+'.tif'
        # Simplest implementation, may crash with very large raster
        MosaicGtifs(outMapFolder,tileTifs,mosaicTifName,keepTifs=False)
    
    # check time
    endTime = time.time()
    usedTime = round((endTime-startTime)/60,3)
    libTime[libName] = usedTime
    # print(f'{libName} processing time (minutes):', usedTime)

# Show processing time
# Individual lib processing time
print('Individual library mapping time:', libTime)
# total time
endTimeAllLibs = time.time()
print('Total processing time (minutes):', round((endTimeAllLibs-startTimeAllLibs)/60,3))

Tiled FLDPLN library folder: E:/fldpln/workshop/devcon26/wildcat_10m_3dep
Map folder: E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\mindtf
Libraries to map: ['tiledlib']
Map tiledlib ...
Tiles need to be mapped: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Actual mapped tiles: ['E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_1.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_2.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_3.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_4.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_5.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_6.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_7.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mindtf\\tiledlib_tile_8.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3

## Visualize and Examine the Special Maps in GIS

Here we can visualize the special maps generated above using GIS software such as ArcGIS Pro to better understand them.